In [2]:
import pandas as pd

In [ ]:
''' import os

print("Noteboofolder:")
print(os.getcwd())

print("\nFiles:")
print(os.listdir())  '''

' import os\n\nprint("Notebook working folder:")\nprint(os.getcwd())\n\nprint("\nFiles in this folder:")\nprint(os.listdir())  '

# Running the preprocessed file

In [4]:
%run "Main_Preprocessing_1.py"

C:\Users\shibu\OneDrive\Desktop\ALLAN\Projects in Data Science\GDSC\TRIAL for me\Main_Preprocessing_1.py:9: DtypeWarning: Columns (0: Recurrent Gain Loss, 1: Genes in Segment) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv("genomic_features.csv")


Data loaded:
  df shape : (242035, 19)
  df2 shape: (698000, 9)
  df3 shape: (2266, 96)
  df4 shape: (2132, 49)

After MSI and Growth fill:
  MSI missing         : 630
  Growth Prop missing : 0

After tissue descriptor fills:
  GDSC Tissue descriptor 1 missing: 0
  GDSC Tissue descriptor 2 missing: 0

After feature column cleaning — missing counts:
TARGET                                     27872
DRUG_NAME                                      0
TCGA_DESC                                   1067
Microsatellite instability Status (MSI)      630
Growth Properties                              0
GDSC Tissue descriptor 1                       0
GDSC Tissue descriptor 2                       0

TCGA Stage 1 fill results:
  Missing before                             : 1067
  Filled from cancer-type column             : 360
  Filled manually                            : 707
  Filled from Tissue descriptor 2 (unique)  : 0
  Missing after                              : 0

TCGA Stage 2 (OTHER -> PRA

# Feature Selection

Baseline Feature Selection and Model Comparison

In [5]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import VarianceThreshold, mutual_info_regression

# Start with the scaled training and test sets from preprocessing
X_train_fs_base = X_train_scaled.copy()
X_test_fs_base = X_test_scaled.copy()

print("Starting feature-selection matrices:")
print("X_train_fs_base:", X_train_fs_base.shape)
print("X_test_fs_base :", X_test_fs_base.shape)

# Step 1: remove constant features
vt = VarianceThreshold(threshold=0.0)

X_train_vt_array = vt.fit_transform(X_train_fs_base)
X_test_vt_array = vt.transform(X_test_fs_base)

selected_cols_after_vt = X_train_fs_base.columns[vt.get_support()]

X_train_vt = pd.DataFrame(
    X_train_vt_array,
    columns=selected_cols_after_vt,
    index=X_train_fs_base.index
)

X_test_vt = pd.DataFrame(
    X_test_vt_array,
    columns=selected_cols_after_vt,
    index=X_test_fs_base.index
)

print("\nAfter removing constant features:")
print("X_train_vt:", X_train_vt.shape)
print("X_test_vt :", X_test_vt.shape)

# Step 2: rank remaining features using mutual information
mi_scores = mutual_info_regression(
    X_train_vt,
    y_train,
    random_state=42
)

mi_df = pd.DataFrame({
    "Feature": X_train_vt.columns,
    "MI_Score": mi_scores
}).sort_values("MI_Score", ascending=False).reset_index(drop=True)

print("\nTop 25 selected features by mutual information:")
display(mi_df.head(25))

Starting feature-selection matrices:
X_train_fs_base: (193628, 146)
X_test_fs_base : (48407, 146)

After removing constant features:
X_train_vt: (193628, 146)
X_test_vt : (48407, 146)

Top 25 selected features by mutual information:


,Feature,MI_Score
0,DRUG_NAME_target_enc,0.643149
1,ploidy_wes,0.112369
2,ploidy_snp6,0.111351
3,mutational_burden,0.078163
4,Growth Properties_Suspension,0.028242
5,TARGET_PATHWAY_Mitosis,0.026173
6,Growth Properties_Adherent,0.021460
7,GDSC Tissue descriptor 1_leukemia,0.017116
8,TARGET_PATHWAY_Metabolism,0.014283
9,GDSC Tissue descriptor 1_lymphoma,0.014073


In [ ]:
'''mi_df.to_csv("feature_selection_mi_ranking.csv", index=False)
print("Saved: feature_selection_mi_ranking.csv")
'''

Saved: feature_selection_mi_ranking.csv


In [10]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [11]:
# Different feature-set sizes to compare
top_100_features = mi_df.head(100)["Feature"].tolist()
top_75_features  = mi_df.head(75)["Feature"].tolist()
top_50_features  = mi_df.head(50)["Feature"].tolist()
# Full non-constant feature set
X_train_all = X_train_vt.copy()
X_test_all = X_test_vt.copy()
# Reduced feature sets
X_train_top100 = X_train_vt[top_100_features].copy()
X_test_top100 = X_test_vt[top_100_features].copy()
X_train_top75 = X_train_vt[top_75_features].copy()
X_test_top75 = X_test_vt[top_75_features].copy()
X_train_top50 = X_train_vt[top_50_features].copy()
X_test_top50 = X_test_vt[top_50_features].copy()

print("Feature-set shapes ready for modeling:")
print("All features :", X_train_all.shape, X_test_all.shape)

def evaluate_feature_set(name, Xtr, Xte, ytr, yte):
    model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.05,
        depth=8,
        loss_function="RMSE",
        eval_metric="RMSE",
        verbose=0,
        random_seed=42                       )

    model.fit(Xtr, ytr)
    preds = model.predict(Xte)
    rmse = np.sqrt(mean_squared_error(yte, preds))
    mae = mean_absolute_error(yte, preds)
    r2 = r2_score(yte, preds)
    return {
        "Feature_Set": name,
        "Num_Features": Xtr.shape[1],
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2           }

results = []
results.append(evaluate_feature_set("All features", X_train_all, X_test_all, y_train, y_test))
results.append(evaluate_feature_set("Top 100 MI", X_train_top100, X_test_top100, y_train, y_test))
results.append(evaluate_feature_set("Top 75 MI", X_train_top75, X_test_top75, y_train, y_test))
results.append(evaluate_feature_set("Top 50 MI", X_train_top50, X_test_top50, y_train, y_test))
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)

display(results_df)

Feature-set shapes ready for modeling:
All features : (193628, 146) (48407, 146)


,Feature_Set,Num_Features,RMSE,MAE,R2
0,All features,146,1.228701,0.918174,0.802114
1,Top 100 MI,100,1.229113,0.918343,0.801981
2,Top 75 MI,75,1.231896,0.919849,0.801083
3,Top 50 MI,50,1.247425,0.928416,0.796036


In [12]:
'''     
results_df.to_csv("feature_selection_model_comparison.csv", index=False)
print("Saved: feature_selection_model_comparison.csv")
'''

'     \nresults_df.to_csv("feature_selection_model_comparison.csv", index=False)\nprint("Saved: feature_selection_model_comparison.csv")\n'

Mutual Info Regression

In [13]:
from sklearn.feature_selection import mutual_info_regression

# Use the non-constant feature set from the earlier baseline step
# If you already created X_train_vt, this will use it directly
X_fs = X_train_vt.copy()
y_fs = y_train.copy()

# Settings for repeated subsampling
n_runs = 5
sample_size = 40000      # adjust if needed
top_k = 50               # number of top features to keep per run
random_seeds = [11, 22, 33, 44, 55]

feature_names = X_fs.columns.tolist()

# To store ranking results from each run
all_run_rankings = []
selection_counts = pd.Series(0, index=feature_names, dtype=int)
mi_score_matrix = pd.DataFrame(index=feature_names)

print("Running repeated mutual information feature selection...")
print(f"Training matrix shape: {X_fs.shape}")
print(f"Number of runs: {n_runs}")
print(f"Sample size per run: {sample_size}")
print(f"Top features kept per run: {top_k}")

for i, seed in enumerate(random_seeds[:n_runs], start=1):
    # Take a random subsample from the training set
    sample_idx = X_fs.sample(n=sample_size, random_state=seed).index
    X_sub = X_fs.loc[sample_idx]
    y_sub = y_fs.loc[sample_idx]

    # Run mutual information on the subsample
    mi_scores = mutual_info_regression(
        X_sub,
        y_sub,
        random_state=seed
    )

    run_df = pd.DataFrame({
        "Feature": feature_names,
        "MI_Score": mi_scores
    }).sort_values("MI_Score", ascending=False).reset_index(drop=True)

    run_df["Run"] = i
    all_run_rankings.append(run_df)

    # Store scores for later averaging
    mi_score_matrix[f"Run_{i}"] = pd.Series(mi_scores, index=feature_names)

    # Count which features appear in the top_k for this run
    top_features_this_run = run_df.head(top_k)["Feature"].tolist()
    selection_counts.loc[top_features_this_run] += 1

    print(f"\nRun {i} complete")
    print(f"Subsample shape: {X_sub.shape}")
    print("Top 10 features from this run:")
    display(run_df.head(10))

Running repeated mutual information feature selection...
Training matrix shape: (193628, 146)
Number of runs: 5
Sample size per run: 40000
Top features kept per run: 50

Run 1 complete
Subsample shape: (40000, 146)
Top 10 features from this run:


,Feature,MI_Score,Run
0,DRUG_NAME_target_enc,0.629985,1
1,ploidy_wes,0.053735,1
2,ploidy_snp6,0.047129,1
3,mutational_burden,0.029997,1
4,TARGET_PATHWAY_Mitosis,0.026083,1
5,Growth Properties_Suspension,0.023252,1
6,Growth Properties_Adherent,0.015490,1
7,TARGET_PATHWAY_Metabolism,0.014720,1
8,GDSC Tissue descriptor 1_leukemia,0.014438,1
9,GDSC Tissue descriptor 1_lymphoma,0.013153,1



Run 2 complete
Subsample shape: (40000, 146)
Top 10 features from this run:


,Feature,MI_Score,Run
0,DRUG_NAME_target_enc,0.629003,2
1,ploidy_wes,0.055156,2
2,ploidy_snp6,0.044345,2
3,TARGET_PATHWAY_Mitosis,0.026519,2
4,Growth Properties_Suspension,0.024300,2
5,mutational_burden,0.023629,2
6,Growth Properties_Adherent,0.017873,2
7,TARGET_PATHWAY_Protein stability and degradation,0.015588,2
8,GDSC Tissue descriptor 1_leukemia,0.015169,2
9,TARGET_PATHWAY_Other,0.014851,2



Run 3 complete
Subsample shape: (40000, 146)
Top 10 features from this run:


,Feature,MI_Score,Run
0,DRUG_NAME_target_enc,0.628691,3
1,ploidy_wes,0.047950,3
2,ploidy_snp6,0.044703,3
3,mutational_burden,0.031154,3
4,TARGET_PATHWAY_Mitosis,0.027992,3
5,Growth Properties_Suspension,0.025327,3
6,Growth Properties_Adherent,0.020811,3
7,TARGET_PATHWAY_Metabolism,0.015719,3
8,TARGET_PATHWAY_Protein stability and degradation,0.013775,3
9,TARGET_PATHWAY_Other,0.012948,3



Run 4 complete
Subsample shape: (40000, 146)
Top 10 features from this run:


,Feature,MI_Score,Run
0,DRUG_NAME_target_enc,0.619130,4
1,ploidy_wes,0.051599,4
2,ploidy_snp6,0.036484,4
3,TARGET_PATHWAY_Mitosis,0.025391,4
4,Growth Properties_Suspension,0.024414,4
5,mutational_burden,0.024032,4
6,Growth Properties_Adherent,0.018737,4
7,GDSC Tissue descriptor 1_leukemia,0.014691,4
8,TARGET_PATHWAY_Metabolism,0.013739,4
9,GDSC Tissue descriptor 1_lymphoma,0.013435,4



Run 5 complete
Subsample shape: (40000, 146)
Top 10 features from this run:


,Feature,MI_Score,Run
0,DRUG_NAME_target_enc,0.625357,5
1,ploidy_wes,0.043265,5
2,ploidy_snp6,0.039609,5
3,TARGET_PATHWAY_Mitosis,0.026928,5
4,mutational_burden,0.023936,5
5,Growth Properties_Suspension,0.023557,5
6,Growth Properties_Adherent,0.020680,5
7,TARGET_PATHWAY_Metabolism,0.014438,5
8,GDSC Tissue descriptor 1_leukemia,0.013741,5
9,TARGET_PATHWAY_Other,0.013596,5


In [14]:
stability_df = pd.DataFrame({
    "Feature": feature_names,
    "Times_Selected": selection_counts.values,
    "Selection_Rate": selection_counts.values / n_runs,
    "Average_MI_Score": mi_score_matrix.mean(axis=1).values
})

stability_df = stability_df.sort_values(
    by=["Times_Selected", "Average_MI_Score"],
    ascending=[False, False]
).reset_index(drop=True)

print("Top stable features across repeated subsamples:")
display(stability_df.head(25))

Top stable features across repeated subsamples:


,Feature,Times_Selected,Selection_Rate,Average_MI_Score
0,DRUG_NAME_target_enc,5,1.0,0.626433
1,ploidy_wes,5,1.0,0.050341
2,ploidy_snp6,5,1.0,0.042454
3,TARGET_PATHWAY_Mitosis,5,1.0,0.026582
4,mutational_burden,5,1.0,0.026550
5,Growth Properties_Suspension,5,1.0,0.024170
6,Growth Properties_Adherent,5,1.0,0.018718
7,TARGET_PATHWAY_Metabolism,5,1.0,0.014378
8,GDSC Tissue descriptor 1_leukemia,5,1.0,0.014118
9,TARGET_PATHWAY_Protein stability and degradation,5,1.0,0.013321


In [16]:
stability_df = pd.DataFrame({
    "Feature": feature_names,
    "Times_Selected": selection_counts.values,
    "Selection_Rate": selection_counts.values / n_runs,
    "Average_MI_Score": mi_score_matrix.mean(axis=1).values
})

stability_df = stability_df.sort_values(
    by=["Times_Selected", "Average_MI_Score"],
    ascending=[False, False]
).reset_index(drop=True)

print("Top stable features across repeated subsamples:")
display(stability_df.head(25))


Top stable features across repeated subsamples:


,Feature,Times_Selected,Selection_Rate,Average_MI_Score
0,DRUG_NAME_target_enc,5,1.0,0.626433
1,ploidy_wes,5,1.0,0.050341
2,ploidy_snp6,5,1.0,0.042454
3,TARGET_PATHWAY_Mitosis,5,1.0,0.026582
4,mutational_burden,5,1.0,0.026550
5,Growth Properties_Suspension,5,1.0,0.024170
6,Growth Properties_Adherent,5,1.0,0.018718
7,TARGET_PATHWAY_Metabolism,5,1.0,0.014378
8,GDSC Tissue descriptor 1_leukemia,5,1.0,0.014118
9,TARGET_PATHWAY_Protein stability and degradation,5,1.0,0.013321
